<a href="https://colab.research.google.com/github/iigaa/dlai/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install mergekit from the repo
%pip install git+https://github.com/arcee-ai/mergekit.git

# Install necessary Hugging Face libraries
%pip install transformers accelerate huggingface_hub

In [ ]:
# To use uncensored models, access from the profile might be required
from huggingface_hub import login

login(token="...")

In [ ]:
import yaml

config = """
models:
  - model: Qwen/Qwen2-1.5B-Instruct
  - model: dphn/dolphin-2.9.3-qwen2-1.5b
merge_method: slerp
base_model: Qwen/Qwen2-1.5B-Instruct
parameters:
  t:
    - value: 1
dtype: float16
tokenizer_source: base
"""

with open('config.yaml', 'w') as f:
    f.write(config)

print("Configuration saved as config.yaml")

In [ ]:
!mergekit-yaml config.yaml ./merged-model --cuda --copy-tokenizer --lazy-unpickle

In [ ]:
# Install a helper library for loading models onto the GPU
!pip install accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./merged-model"

print("Loading the tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path)

print("Loading the Merged Model...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Red teaming
test_prompt = "..."

messages = [
    # {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": test_prompt}
]

# Format the prompt for the Qwen architecture
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to("cuda")

print("Generating response...")
outputs = model.generate(
    inputs.input_ids,
    max_new_tokens=300,
    temperature=0.7,
    do_sample=True
)

# Decode the output and slice off the prompt
response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

print("\n-------------------------")
print("--- AI RESPONSE ---")
print("-------------------------\n")
print(response)

In [ ]:
from transformers import pipeline

print("Loading the Toxicity Scorer...")
# Lightweight model trained to detect toxic, hostile, or insulting text
scorer = pipeline("text-classification", model="unitary/toxic-bert")

# Outputs from a given prompt
responses = {
    "t = 0.00": "...",
    "t = 0.25": "...",
    "t = 0.50": "...",
    "t = 0.75": "...",
    "t = 1.00": "..."
}

print("\n--- TOXICITY SCORES ---")
for t_value, text in responses.items():
    # Slice the text to the first 500 characters just to make it fast
    result = scorer(text[:500])[0]

    # Turn the raw toxic-bert score to see how high the "toxic" probability is
    score = result['score'] if result['label'] == 'toxic' else 1 - result['score']

    print(f"{t_value}: {score:.4f}")